# 迹线 Transformer 涡提取 —— Kaggle 训练（票 07）

在本 Notebook 完成 `pipedcylinder2d.nc` 上 200 epoch 全量训练（Kaggle T4×2、12h 会话硬上限 → ≤8h 分块 + 每 epoch checkpoint + 跨会话断点续训）。

**运行前准备（详见 `kaggle/README.md`）**：
1. Notebook Settings：Accelerator = GPU T4 x2、Internet = ON；
2. Add Input：Dataset A（`kaggle/prepare_dataset_a.py` 打包产物，含 `dataset/meta.json`）；
3.（可选，模式 A）Add-ons → Secrets 添加 `KAGGLE_USERNAME`/`KAGGLE_KEY`，并在下方 `CKPT_DATASET_SLUG` 填 checkpoint 数据集 slug（跨会话自动发布/下载）；
4.（可选，模式 B）把上一会话的 checkpoint 打包文件上传为 Kaggle Dataset 并 Add Input，或从 `/kaggle/working` 下载后重新上传。

**cell 直接顺序执行（Run All）**。会话结束（12h）后重启会话再 Run All 即从 latest checkpoint 续训。

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "h5py", "PyYAML", "matplotlib", "tqdm", "kaggle"], check=True)
import sys, torch, numpy, h5py, yaml
print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())

In [ ]:
import os, pathlib
REPO_URL = "https://github.com/ziyixu317-wq/2d-vortex-extraction-260825.git"
REPO_DIR = "/kaggle/working/repo"
if os.path.isdir(REPO_DIR) and (pathlib.Path(REPO_DIR) / "train_kaggle.py").exists():
    print("仓库已就绪")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)   # Script/Notebook 双保险（import dataset 等）
print("repo 根文件:", sorted(os.listdir(REPO_DIR))[:12])
print("cwd:", os.getcwd())

In [ ]:
import glob, shutil, zipfile
CONFIG = "config/pathline_transformer_cylinder.yaml"
TRAIN_CONFIG = CONFIG              # 训练配置（块 5 校准可能生成优化的覆盖配置）
cfg0 = yaml.safe_load(open(CONFIG, encoding="utf-8"))
TOTAL_EPOCHS = int(cfg0["train"]["epochs"])     # 200（HANDOFF §6；以 config 为单一来源）
CHUNK_BUDGET_H = 7.5              # 每块会话预算（12h 上限内留自检/打包余量）
CKPT_DATASET_SLUG = ""           # 模式 A：checkpoint 数据集 slug（如 "yourname/vortex-train-ckpt"）；留空 = 手动模式

# ---- 输入探测（布局自适应：直接 / 多一层嵌套 / 任意深度 / zip 未解压回退）
def find_input(marker, root="/kaggle/input"):
    """在挂载树中定位含 marker 的挂载根目录（返回该根 Path；含 marker 时不得 None）。"""
    for d in sorted(pathlib.Path(root).glob("*")):
        if (d / marker).exists():
            return d
    for hit in pathlib.Path(root).rglob(marker):
        return hit.parents[len(marker.split("/")) - 1]   # 任意深度 → 回到挂载根
    return None

print("[env] /kaggle/input 挂载:", [p.name for p in pathlib.Path("/kaggle/input").glob("*")])
ds_in = find_input("dataset/meta.json")
if ds_in is None:
    zips = list(pathlib.Path("/kaggle/input").rglob("*.zip"))
    if zips:
        print(f"[env] 未直接命中；检测到 zip（未解压?）→ 解压后重找: {[z.name for z in zips]}")
        ex = pathlib.Path("/kaggle/working/dataset_a_extract")
        ex.mkdir(parents=True, exist_ok=True)
        for z in zips:
            with zipfile.ZipFile(z) as zf:
                zf.extractall(ex)
        ds_in = find_input("dataset/meta.json", root=str(ex))
if ds_in is None:
    print("[env] 仍未找到 dataset/meta.json。输入树（/kaggle/input 全览）：")
    for d in pathlib.Path("/kaggle/input").rglob("*"):
        print("  ", d)
    raise AssertionError("未找到 Dataset A（应含 dataset/meta.json）——检查 Add Input 挂载与打包内容")

# ---- Dataset A（nc + prepare_dataset 产物）→ outputs/dataset
dst = pathlib.Path("outputs/dataset")
if dst.exists():
    shutil.rmtree(dst)
dst.parent.mkdir(parents=True, exist_ok=True)
if (ds_in / "dataset").is_symlink() is False and (ds_in / "dataset").exists():
    try:
        os.symlink(ds_in / "dataset", dst)
        print("数据已链接 →", ds_in / "dataset")
    except OSError:
        shutil.copytree(ds_in / "dataset", dst)
        print("数据已复制 →", dst)

# ---- checkpoint 数据集（跨会话续训；模式 A 自动下载 / 模式 B 手动挂载 input）
if CKPT_DATASET_SLUG:
    subprocess.run(["kaggle", "datasets", "download", CKPT_DATASET_SLUG,
                    "-p", "/tmp/ckpt", "--unzip"], check=True)
    ckpt_src = pathlib.Path("/tmp/ckpt")
else:
    ckpt_src = find_input("pathline_transformer_cylinder_ckpt_latest.pth")
if ckpt_src is not None:
    os.makedirs("outputs/train", exist_ok=True)
    for p in pathlib.Path(ckpt_src).glob("*"):
        if p.is_file():
            shutil.copy2(p, pathlib.Path("outputs/train") / p.name)
    print("checkpoint 已还原 → outputs/train/")
else:
    print("首次会话：无 checkpoint，从零开始（--resume auto 自动处理）")


In [ ]:
# 验收 1：Notebook 环境 import vendor + 数据加载通过（失败会 raise，不静默）
subprocess.run([sys.executable, "kaggle/self_check.py",
                "--data-root", "outputs/dataset",
                "--config", CONFIG, "--device", "cuda", "--n-samples", "4"], check=True, cwd=REPO_DIR)

In [ ]:
# 验收 2：1 epoch 实测步速（首会话真测；后续会话复用 bench_info.json，不重跑校准）
import time, json, os
BENCH_INFO = "outputs/bench_info.json"
if os.path.exists(BENCH_INFO):
    bench = json.load(open(BENCH_INFO, encoding="utf-8"))
    print(f"复用步速基准: 1 epoch = {bench['seconds_per_epoch']:.1f} s（{bench['timestamp']} 实测）")
else:
    BENCH_YAML = "outputs/bench_config.yaml"
    cfg = yaml.safe_load(open(CONFIG, encoding="utf-8"))
    cfg["train"]["epochs"] = 1
    cfg["train"]["ckpt_dir"] = "outputs/bench"
    cfg["train"]["run_name"] = "bench"
    yaml.safe_dump(cfg, open(BENCH_YAML, "w", encoding="utf-8"), allow_unicode=True)
    t0 = time.time()
    subprocess.run([sys.executable, "train_kaggle.py", "--config", BENCH_YAML,
                    "--resume", "none"], check=True, cwd=REPO_DIR)
    seconds_per_epoch = time.time() - t0
    bench = {"seconds_per_epoch": seconds_per_epoch,
             "samples_per_epoch": cfg["data"]["samples_per_epoch"],
             "steps_per_epoch": int(cfg["data"]["samples_per_epoch"] / cfg["data"]["batch_size"]),
             "timestamp": time.strftime("%Y-%m-%d %H:%M")}
    json.dump(bench, open(BENCH_INFO, "w", encoding="utf-8"), indent=2)
    print(f"1 epoch 实测 = {seconds_per_epoch:.1f} s（{bench['steps_per_epoch']} 步，40000 样本）")

# ---- 校准：总时长预算检查（HANDOFF §7 风险预案：超预算启用 DataParallel/AMP/降样本数）
sps = float(bench["seconds_per_epoch"])
total_h = sps * TOTAL_EPOCHS / 3600
print(f"200 epoch 预计总时长 ≈ {total_h:.1f} h（目标 ≤ 4 个 12h 会话）")
if total_h > 4 * 12:
    opt = yaml.safe_load(open(CONFIG, encoding="utf-8"))
    opt["train"]["amp"] = True
    opt["train"]["data_parallel"] = torch.cuda.device_count() > 1
    n_samples = max(20000, int(opt["data"]["samples_per_epoch"] * 0.5))
    opt["data"]["samples_per_epoch"] = n_samples     # HANDOFF §6 下限 20000
    yaml.safe_dump(opt, open("outputs/train_opt.yaml", "w", encoding="utf-8"),
                    allow_unicode=True)
    TRAIN_CONFIG = "outputs/train_opt.yaml"
    print(f"超预算：已生成 {TRAIN_CONFIG}（AMP + DataParallel + 样本数→{n_samples}）")
    print("回填 HANDOFF §6：以实测校准 samples_per_epoch（下限 20000）；步速/时长见本 cell")
else:
    TRAIN_CONFIG = CONFIG
    print("预算内：直接用生产配置")

from kaggle.chunking import plan_chunks
plan = plan_chunks(TOTAL_EPOCHS, sps, CHUNK_BUDGET_H * 3600)
print(f"{CHUNK_BUDGET_H}h 预算 → 每块最多 {max(plan)} epoch；分块计划 {plan}（约 {len(plan)} 个会话）")
print(f"本块训练配置: {TRAIN_CONFIG}")

In [ ]:
# 验收 3：分块训练（每会话一块；--resume auto 从 latest 无损伤续训）
import torch, pathlib as _pl
from kaggle.chunking import plan_chunks
bench = json.load(open("outputs/bench_info.json", encoding="utf-8"))
sps = float(bench["seconds_per_epoch"])
latest = _pl.Path("outputs/train/pathline_transformer_cylinder_ckpt_latest.pth")
def progress():
    if not latest.exists():
        return 0
    return int(torch.load(latest, map_location="cpu")["epoch"]) + 1

p = progress()
assert p < TOTAL_EPOCHS, f"训练已完成（已到 {p} epoch）——直接看收尾 cell"
plan = plan_chunks(TOTAL_EPOCHS - p, sps, CHUNK_BUDGET_H * 3600)
chunk = plan[0]
target = p + chunk
cmd = [sys.executable, "train_kaggle.py", "--config", TRAIN_CONFIG,
       "--resume", "auto", "--epochs", str(target)]
if target >= TOTAL_EPOCHS:
    cmd.append("--report-f1")      # 最后一块：训练完成后记录 val F1（验收 4）
print(f"[分块] 从 epoch {p} 续到 {target}（本块 {chunk}；完整计划 {plan}）")
subprocess.run(cmd, check=True, cwd=REPO_DIR)
print(f"[分块] 本块完成：此会话结束（进度 {progress()}/{TOTAL_EPOCHS}）。"
      "重启会话再次 Run All 即续训）")

In [ ]:
# 块尾：checkpoint 打包（跨会话续训载体）
import zipfile
srcdir = pathlib.Path("outputs/train")
zpath = pathlib.Path("/kaggle/working") / "ckpt_snapshot.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(srcdir.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print("checkpoint 快照已打包:", zpath, f"({zpath.stat().st_size/1e6:.1f} MB)")
if CKPT_DATASET_SLUG:
    # 模式 A：发布为 checkpoint 数据集新版本（下次会话自动下载续训）
    subprocess.run(["kaggle", "datasets", "version", CKPT_DATASET_SLUG,
                    "-p", str(srcdir), "--dir-mode", "zip",
                    "-m", f"train chunk to epoch {progress()-1}"], check=True)
    print("已发布 checkpoint 数据集新版本 →", CKPT_DATASET_SLUG)
else:
    # 模式 B：手动 —— 下载 zpath，上传为 Kaggle Dataset（挂载为 input 供下次会话）
    print("手动模式：下载 ckpt_snapshot.zip → Kaggle 新建 Dataset 上传 → 下次会话 Add Input")

In [ ]:
# 验收 4：训练完成收尾（val F1 记录 + 最终 checkpoint 归档）
f1_path = pathlib.Path("outputs/train/pathline_transformer_cylinder_val_f1.json")
if f1_path.exists():
    print("val F1 记录（自然分布）:")
    print(json.dumps(json.loads(f1_path.read_text(encoding="utf-8")), indent=2))
else:
    print("尚未训练完成（无 val_f1.json）——继续跑分块 cell 直至 200 epoch")

import hashlib
final_zip = pathlib.Path("/kaggle/working") / "final_ckpt.zip"
with zipfile.ZipFile(final_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(srcdir.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
h = hashlib.sha256(final_zip.read_bytes()).hexdigest()
print(f"最终 checkpoint 归档: {final_zip} ({final_zip.stat().st_size/1e6:.1f} MB)\nsha256 = {h}")
print("步骤：下载 final_ckpt.zip → 本地 outputs/archive/ → 回填票文件（val F1/步速/checkpoint 位置）")